# Joint Kinematics Dimensionality Reduction Analysis

This notebook analyzes walking kinematics through dimensionality reduction (PCA/UMAP) to understand how joints coordinate during free-walking behavior.

**Data source**: JARVIS 3D tracking + MuJoCo inverse kinematics

**Reference**: Analysis patterns from Grant's fly_walking_analysis and Pratt et al., 2024

## 1. Configuration

In [ ]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from scipy import signal
from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Optional: UMAP
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not available. Install with: pip install umap-learn")

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 10

In [ ]:
# =============================================================================
# CONFIGURATION - Modify these parameters as needed
# =============================================================================

# Frame rate
FPS = 800  # Hz

# Joint sets for analysis (choose one)
JOINT_SETS = {
    'core': ['coxa_flexion', 'coxa', 'femur', 'tibia'],  # 4 per leg = 24 total
    'main': ['coxa_flexion', 'coxa_twist', 'coxa', 'femur_twist', 'femur', 'tibia', 'tarsus'],  # 7 per leg = 42 total
    'full': ['coxa_flexion', 'coxa_twist', 'coxa', 'femur_twist', 'femur', 'tibia', 
             'tarsus', 'tarsus2', 'tarsus3', 'tarsus4', 'tarsus5'],  # 11 per leg = 66 total
}

# Active joint set - change this to switch between sets
ACTIVE_JOINT_SET = 'core'

# Leg naming convention
LEGS = ['T1_left', 'T1_right', 'T2_left', 'T2_right', 'T3_left', 'T3_right']
LEG_LABELS = {'T1_left': 'L1', 'T1_right': 'R1', 
              'T2_left': 'L2', 'T2_right': 'R2',
              'T3_left': 'L3', 'T3_right': 'R3'}  # For plotting

# Joint name mapping to reference literature
JOINT_LABELS = {
    'coxa_flexion': 'Coxa Flex',
    'coxa_twist': 'Coxa Rot',
    'coxa': 'Coxa Abd',
    'femur_twist': 'Femur Rot',
    'femur': 'Femur Flex',
    'tibia': 'Tibia Flex',
    'tarsus': 'Tarsus Flex',
}

# Data paths - UPDATE THESE
H5_PATH = Path('/home/user/src/JARVIS-HybridNet/projects/fly50_V4/predictions/predictions3D/Predictions_3D_20260121-104455/Fruitfly_ik_V1_free.h5')

# Output directory
OUTPUT_DIR = Path('./output/joint_kinematics')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Print configuration
print(f"Configuration loaded.")
print(f"Frame rate: {FPS} Hz")
print(f"Active joint set: '{ACTIVE_JOINT_SET}' ({len(JOINT_SETS[ACTIVE_JOINT_SET])} joints/leg = {len(JOINT_SETS[ACTIVE_JOINT_SET]) * len(LEGS)} total)")
print(f"Joints: {JOINT_SETS[ACTIVE_JOINT_SET]}")
print(f"H5 path: {H5_PATH}")
print(f"Output: {OUTPUT_DIR.absolute()}")

## 2. Data Loading Functions

In [ ]:
def load_ik_data(h5_path):
    """
    Load inverse kinematics data from H5 file.
    
    Returns dict with:
        - qpos: (N, 93) joint angles
        - qvel: (N, 92) joint velocities
        - xpos: (N, 68, 3) body positions
        - names_qpos: list of joint names
        - names_xpos: list of body names
        - kp_names: list of keypoint names
        - marker_sites: (N, 50, 3) keypoint positions
    """
    with h5py.File(h5_path, 'r') as f:
        data = {
            'qpos': f['qpos'][:],
            'qvel': f['qvel'][:],
            'xpos': f['xpos'][:],
            'xquat': f['xquat'][:],
            'marker_sites': f['marker_sites'][:],
            'names_qpos': [n.decode() for n in f['names_qpos'][:]],
            'names_xpos': [n.decode() for n in f['names_xpos'][:]],
            'kp_names': [n.decode() for n in f['kp_names'][:]],
        }
    
    print(f"Loaded IK data: {data['qpos'].shape[0]} frames")
    print(f"  qpos: {data['qpos'].shape} ({len(data['names_qpos'])} joints)")
    print(f"  xpos: {data['xpos'].shape} ({len(data['names_xpos'])} bodies)")
    print(f"  marker_sites: {data['marker_sites'].shape} ({len(data['kp_names'])} keypoints)")
    
    return data


def build_joint_index(names_qpos, joint_set, legs):
    """
    Build index mapping from joint set to qpos columns.
    
    Returns:
        dict: {joint_name: index in qpos}
        list: ordered list of (leg, joint, index) tuples
    """
    joint_index = {}
    joint_list = []
    
    for leg in legs:
        for joint in joint_set:
            # Construct full joint name as it appears in names_qpos
            full_name = f"{joint}_{leg}"
            
            if full_name in names_qpos:
                idx = names_qpos.index(full_name)
                joint_index[full_name] = idx
                joint_list.append((leg, joint, idx))
            else:
                print(f"Warning: Joint '{full_name}' not found in qpos names")
    
    print(f"Built index for {len(joint_list)} joints")
    return joint_index, joint_list


def extract_joint_angles(qpos, joint_list, frame_range=None):
    """
    Extract joint angles for selected joints into a DataFrame.
    
    Args:
        qpos: (N, 93) array of joint angles
        joint_list: list of (leg, joint, index) tuples
        frame_range: optional (start, end) tuple to slice frames
    
    Returns:
        DataFrame with columns like 'T1_left_coxa_flexion', etc.
    """
    if frame_range is not None:
        start, end = frame_range
        qpos = qpos[start:end+1]
    
    data = {}
    data['frame'] = np.arange(len(qpos))
    
    for leg, joint, idx in joint_list:
        col_name = f"{leg}_{joint}"
        data[col_name] = qpos[:, idx]
    
    return pd.DataFrame(data)


def extract_body_positions(xpos, names_xpos, body_names, frame_range=None):
    """
    Extract 3D positions for specified bodies.
    
    Args:
        xpos: (N, 68, 3) body positions
        names_xpos: list of body names
        body_names: list of bodies to extract
        frame_range: optional (start, end) tuple
    
    Returns:
        DataFrame with columns like 'thorax_x', 'thorax_y', 'thorax_z', etc.
    """
    if frame_range is not None:
        start, end = frame_range
        xpos = xpos[start:end+1]
    
    data = {}
    data['frame'] = np.arange(len(xpos))
    
    for body in body_names:
        if body in names_xpos:
            idx = names_xpos.index(body)
            data[f"{body}_x"] = xpos[:, idx, 0]
            data[f"{body}_y"] = xpos[:, idx, 1]
            data[f"{body}_z"] = xpos[:, idx, 2]
    
    return pd.DataFrame(data)

## 3. Preprocessing Functions

In [ ]:
def compute_derivatives(df, joint_columns, fps=800):
    """
    Compute first and second derivatives of joint angles using Savitzky-Golay filter.
    
    Args:
        df: DataFrame with joint angle columns
        joint_columns: list of column names to differentiate
        fps: frame rate
    
    Returns:
        DataFrame with added _d1 (velocity) and _d2 (acceleration) columns
    """
    dt = 1.0 / fps
    s = 1.0 / dt  # Scale for first derivative
    s2 = 1.0 / (dt * dt)  # Scale for second derivative
    
    df = df.copy()
    
    for col in joint_columns:
        arr = df[col].values
        
        # Skip if too short
        if len(arr) < 7:
            df[f"{col}_d1"] = np.nan
            df[f"{col}_d2"] = np.nan
            continue
        
        # Savitzky-Golay filter: window=5, polyorder=2
        # This matches the reference analysis
        df[f"{col}_d1"] = signal.savgol_filter(arr, 5, 2, deriv=1) * s
        df[f"{col}_d2"] = signal.savgol_filter(arr, 5, 2, deriv=2) * s2
    
    return df


def compute_phases(df, legs, joint='tibia', fps=800):
    """
    Compute step phases using Hilbert transform with bandpass filter.
    
    Args:
        df: DataFrame with joint angle columns
        legs: list of leg names
        joint: which joint to use for phase (default: tibia)
        fps: frame rate
    
    Returns:
        DataFrame with added _phase columns
    """
    df = df.copy()
    
    # Bandpass filter design
    # For 800 Hz, typical step frequency is 10-30 Hz
    # Normalized frequencies: low = 5/400 = 0.0125, high = 50/400 = 0.125
    nyquist = fps / 2
    low_freq = 5  # Hz
    high_freq = 50  # Hz
    low_norm = low_freq / nyquist
    high_norm = high_freq / nyquist
    
    sos = signal.butter(1, [low_norm, high_norm], 'bandpass', output='sos')
    
    for leg in legs:
        col = f"{leg}_{joint}"
        if col not in df.columns:
            print(f"Warning: Column {col} not found")
            continue
        
        ang = df[col].values
        
        # Skip if too short
        if len(ang) < 50:
            df[f"{leg}_phase"] = np.nan
            continue
        
        # Normalize
        ang_norm = (ang - np.mean(ang)) / (np.std(ang) + 1e-8)
        
        # Apply bandpass filter
        ang_filt = signal.sosfiltfilt(sos, ang_norm)
        
        # Hilbert transform to get phase
        phase = np.angle(signal.hilbert(ang_filt))
        
        # Trim edges (phase is unstable at boundaries)
        n_trim = int(fps * 0.01)  # 10ms at each edge
        phase[:n_trim] = np.nan
        phase[-n_trim:] = np.nan
        
        df[f"{leg}_phase"] = phase
    
    return df


def classify_swing_stance(df, legs, method='phase'):
    """
    Classify swing vs stance for each leg.
    
    Args:
        df: DataFrame with phase columns
        legs: list of leg names
        method: 'phase' (phase >= 0 = swing) or 'velocity'
    
    Returns:
        DataFrame with added _swing_stance columns (1 = stance, 0 = swing)
    """
    df = df.copy()
    
    for leg in legs:
        if method == 'phase':
            phase_col = f"{leg}_phase"
            if phase_col in df.columns:
                # Convention: phase >= 0 is swing, phase < 0 is stance
                df[f"{leg}_swing_stance"] = (df[phase_col] < 0).astype(int)
        # Add velocity-based method if needed
    
    return df


def compute_n_legs_stance(df, legs):
    """
    Compute number of legs in stance phase per frame.
    """
    df = df.copy()
    stance_cols = [f"{leg}_swing_stance" for leg in legs if f"{leg}_swing_stance" in df.columns]
    if stance_cols:
        df['n_legs_stance'] = df[stance_cols].sum(axis=1)
    return df

## 4. Dimensionality Reduction Functions

In [ ]:
def get_feature_matrix(df, joint_list, include_derivatives=True):
    """
    Build feature matrix for dimensionality reduction.
    
    Args:
        df: DataFrame with joint angles (and derivatives if include_derivatives=True)
        joint_list: list of (leg, joint, idx) tuples
        include_derivatives: whether to include _d1 columns
    
    Returns:
        X: (N, features) array
        feature_names: list of feature names
    """
    # Build list of columns to include
    feature_cols = []
    for leg, joint, _ in joint_list:
        col = f"{leg}_{joint}"
        if col in df.columns:
            feature_cols.append(col)
            if include_derivatives and f"{col}_d1" in df.columns:
                feature_cols.append(f"{col}_d1")
    
    # Extract and handle NaNs
    X = df[feature_cols].values
    
    # Remove rows with NaNs
    valid_mask = ~np.any(np.isnan(X), axis=1)
    X = X[valid_mask]
    
    print(f"Feature matrix: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"  Removed {(~valid_mask).sum()} samples with NaNs")
    
    return X, feature_cols, valid_mask


def run_pca(X, n_components=10, standardize=True):
    """
    Run PCA on feature matrix.
    
    Args:
        X: (N, features) array
        n_components: number of PCs to compute
        standardize: whether to standardize features first
    
    Returns:
        pca_result: (N, n_components) transformed data
        pca: fitted PCA object
        scaler: fitted StandardScaler (or None)
        loadings: DataFrame of loadings
    """
    scaler = None
    if standardize:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(X)
    
    print(f"PCA: {n_components} components")
    print(f"  Explained variance: {pca.explained_variance_ratio_[:5].round(3)}...")
    print(f"  Total explained: {pca.explained_variance_ratio_.sum():.1%}")
    
    return pca_result, pca, scaler


def run_umap(X, n_neighbors=15, min_dist=0.1, n_components=2, standardize=True):
    """
    Run UMAP on feature matrix.
    """
    if not HAS_UMAP:
        print("UMAP not available")
        return None, None, None
    
    scaler = None
    if standardize:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=n_components)
    umap_result = reducer.fit_transform(X)
    
    print(f"UMAP: {n_components} components (n_neighbors={n_neighbors}, min_dist={min_dist})")
    
    return umap_result, reducer, scaler

## 5. Visualization Functions

In [ ]:
def plot_pca_variance(pca, ax=None):
    """
    Plot explained variance ratio (scree plot).
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    
    n_pcs = len(pca.explained_variance_ratio_)
    x = np.arange(1, n_pcs + 1)
    
    # Bar plot for individual variance
    ax.bar(x, pca.explained_variance_ratio_, alpha=0.7, label='Individual')
    
    # Line plot for cumulative variance
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    ax.plot(x, cumvar, 'ro-', label='Cumulative')
    
    ax.set_xlabel('Principal Component')
    ax.set_ylabel('Explained Variance Ratio')
    ax.set_title('PCA Explained Variance')
    ax.legend()
    ax.set_xticks(x)
    ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90%')
    
    return ax


def plot_pca_loadings(pca, feature_names, joint_list, n_pcs=4):
    """
    Plot PCA loadings as heatmap organized by leg and joint.
    """
    loadings = pca.components_[:n_pcs].T
    
    fig, ax = plt.subplots(figsize=(10, max(8, len(feature_names) * 0.3)))
    
    # Create labeled feature names for display
    display_names = []
    for name in feature_names:
        # Parse leg_joint or leg_joint_d1
        parts = name.split('_')
        if parts[-1] == 'd1':
            leg = f"{parts[0]}_{parts[1]}"
            joint = '_'.join(parts[2:-1])
            suffix = ' (vel)'
        else:
            leg = f"{parts[0]}_{parts[1]}"
            joint = '_'.join(parts[2:])
            suffix = ''
        
        leg_label = LEG_LABELS.get(leg, leg)
        joint_label = JOINT_LABELS.get(joint, joint)
        display_names.append(f"{leg_label} {joint_label}{suffix}")
    
    im = ax.imshow(loadings, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
    
    ax.set_yticks(np.arange(len(display_names)))
    ax.set_yticklabels(display_names, fontsize=8)
    ax.set_xticks(np.arange(n_pcs))
    ax.set_xticklabels([f'PC{i+1}' for i in range(n_pcs)])
    ax.set_xlabel('Principal Component')
    ax.set_title('PCA Loadings')
    
    plt.colorbar(im, ax=ax, label='Loading')
    plt.tight_layout()
    
    return fig, ax


def plot_pca_trajectory(pca_result, df_valid, color_by='frame', ax=None, s=5, alpha=0.5):
    """
    Plot PC1 vs PC2 trajectory colored by specified variable.
    
    Args:
        pca_result: (N, n_pcs) array
        df_valid: DataFrame aligned with pca_result (after removing NaNs)
        color_by: column name to use for coloring, or 'frame' for time
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    if color_by == 'frame':
        c = np.arange(len(pca_result))
        cmap = 'viridis'
        label = 'Frame'
    elif color_by in df_valid.columns:
        c = df_valid[color_by].values
        cmap = 'coolwarm' if 'phase' in color_by else 'viridis'
        label = color_by
    else:
        c = 'blue'
        cmap = None
        label = None
    
    scatter = ax.scatter(pca_result[:, 0], pca_result[:, 1], c=c, cmap=cmap, s=s, alpha=alpha)
    
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f'PCA Trajectory (colored by {color_by})')
    
    if cmap is not None:
        plt.colorbar(scatter, ax=ax, label=label)
    
    return ax


def plot_joint_vs_phase(df, leg, joint, phase_col=None, n_bins=24, ax=None):
    """
    Plot joint angle vs step phase with binned statistics.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    col = f"{leg}_{joint}"
    if phase_col is None:
        phase_col = f"{leg}_phase"
    
    if col not in df.columns or phase_col not in df.columns:
        print(f"Columns not found: {col}, {phase_col}")
        return ax
    
    # Get valid data
    valid = df[[col, phase_col]].dropna()
    angles = valid[col].values
    phases = valid[phase_col].values
    
    # Bin by phase
    phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    bin_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
    bin_indices = np.digitize(phases, phase_bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    
    # Compute mean and std per bin
    bin_means = np.array([angles[bin_indices == i].mean() if np.any(bin_indices == i) else np.nan 
                         for i in range(n_bins)])
    bin_stds = np.array([angles[bin_indices == i].std() if np.any(bin_indices == i) else np.nan 
                        for i in range(n_bins)])
    
    # Plot
    ax.scatter(phases, angles, alpha=0.1, s=1, c='gray')
    ax.plot(bin_centers, bin_means, 'b-', lw=2, label='Mean')
    ax.fill_between(bin_centers, bin_means - bin_stds, bin_means + bin_stds, alpha=0.3)
    
    ax.set_xlabel('Step Phase (rad)')
    ax.set_ylabel(f'{JOINT_LABELS.get(joint, joint)} (deg)')
    ax.set_title(f'{LEG_LABELS.get(leg, leg)} {JOINT_LABELS.get(joint, joint)} vs Phase')
    ax.set_xlim(-np.pi, np.pi)
    ax.axvline(0, color='r', linestyle='--', alpha=0.5, label='Swing/Stance')
    ax.legend()
    
    return ax


def plot_leg_coordination(df, legs):
    """
    Plot phase relationships between legs.
    """
    phase_cols = [f"{leg}_phase" for leg in legs if f"{leg}_phase" in df.columns]
    n_legs = len(phase_cols)
    
    fig, axes = plt.subplots(n_legs, n_legs, figsize=(12, 12))
    
    for i, col1 in enumerate(phase_cols):
        for j, col2 in enumerate(phase_cols):
            ax = axes[i, j]
            
            if i == j:
                # Diagonal: histogram of phase
                valid = df[col1].dropna()
                ax.hist(valid, bins=30, density=True, alpha=0.7)
                ax.set_xlim(-np.pi, np.pi)
            else:
                # Off-diagonal: phase vs phase scatter
                valid = df[[col1, col2]].dropna()
                ax.scatter(valid[col2], valid[col1], alpha=0.1, s=1)
                ax.set_xlim(-np.pi, np.pi)
                ax.set_ylim(-np.pi, np.pi)
            
            if i == n_legs - 1:
                leg_label = LEG_LABELS.get(phase_cols[j].replace('_phase', ''), phase_cols[j])
                ax.set_xlabel(leg_label)
            if j == 0:
                leg_label = LEG_LABELS.get(phase_cols[i].replace('_phase', ''), phase_cols[i])
                ax.set_ylabel(leg_label)
    
    plt.suptitle('Inter-leg Phase Relationships')
    plt.tight_layout()
    
    return fig, axes

## 6. Coordination Metrics

In [ ]:
def compute_tripod_coordination_strength(df, legs):
    """
    Compute Tripod Coordination Strength (TCS).
    
    Tripod gait: alternating tripods
    - Left tripod: L1, R2, L3 (T1_left, T2_right, T3_left)
    - Right tripod: R1, L2, R3 (T1_right, T2_left, T3_right)
    
    TCS = overlap duration of tripod swing / total tripod duration
    """
    df = df.copy()
    
    # Define tripod groups
    left_tripod = ['T1_left', 'T2_right', 'T3_left']
    right_tripod = ['T1_right', 'T2_left', 'T3_right']
    
    # Check if we have swing_stance for all legs
    ss_cols_left = [f"{leg}_swing_stance" for leg in left_tripod]
    ss_cols_right = [f"{leg}_swing_stance" for leg in right_tripod]
    
    if not all(col in df.columns for col in ss_cols_left + ss_cols_right):
        print("Warning: swing_stance columns missing")
        return df
    
    # Left tripod: all in swing (swing_stance == 0)
    left_all_swing = (df[ss_cols_left] == 0).all(axis=1)
    left_any_swing = (df[ss_cols_left] == 0).any(axis=1)
    
    right_all_swing = (df[ss_cols_right] == 0).all(axis=1)
    right_any_swing = (df[ss_cols_right] == 0).any(axis=1)
    
    # TCS per frame (1 if all in sync, 0 otherwise)
    df['left_tripod_sync'] = left_all_swing.astype(int)
    df['right_tripod_sync'] = right_all_swing.astype(int)
    
    # Compute overall TCS
    left_tcs = left_all_swing.sum() / left_any_swing.sum() if left_any_swing.sum() > 0 else 0
    right_tcs = right_all_swing.sum() / right_any_swing.sum() if right_any_swing.sum() > 0 else 0
    
    print(f"Tripod Coordination Strength:")
    print(f"  Left tripod (L1-R2-L3): {left_tcs:.2%}")
    print(f"  Right tripod (R1-L2-R3): {right_tcs:.2%}")
    
    return df, {'left_tcs': left_tcs, 'right_tcs': right_tcs}


def compute_relative_phase(df, leg1, leg2):
    """
    Compute relative phase between two legs.
    
    Returns phase difference normalized to [-pi, pi]
    """
    phase1_col = f"{leg1}_phase"
    phase2_col = f"{leg2}_phase"
    
    if phase1_col not in df.columns or phase2_col not in df.columns:
        return None
    
    phase_diff = df[phase1_col] - df[phase2_col]
    
    # Wrap to [-pi, pi]
    phase_diff = np.arctan2(np.sin(phase_diff), np.cos(phase_diff))
    
    return phase_diff


def compute_speed(df, body='thorax', fps=800):
    """
    Compute instantaneous speed from body position.
    """
    x_col = f"{body}_x"
    y_col = f"{body}_y"
    
    if x_col not in df.columns or y_col not in df.columns:
        print(f"Position columns not found for {body}")
        return df
    
    df = df.copy()
    
    dx = np.diff(df[x_col].values, prepend=df[x_col].iloc[0])
    dy = np.diff(df[y_col].values, prepend=df[y_col].iloc[0])
    
    displacement = np.sqrt(dx**2 + dy**2)
    df['speed'] = displacement * fps  # mm/s (assuming positions are in mm)
    
    return df

---

## 7. Main Analysis Pipeline

In [ ]:
# Load IK data
ik_data = load_ik_data(H5_PATH)

In [ ]:
# Build joint index for current joint set
joint_index, joint_list = build_joint_index(
    ik_data['names_qpos'], 
    JOINT_SETS[ACTIVE_JOINT_SET], 
    LEGS
)

# Print joint mapping
print("\nJoint index mapping:")
for leg, joint, idx in joint_list[:6]:  # Show first 6
    print(f"  {leg}_{joint} -> qpos[{idx}]")

In [ ]:
# Extract joint angles for all frames (or specify frame_range for a bout)
# Example: frame_range=(1000, 1500) for a specific bout
df = extract_joint_angles(ik_data['qpos'], joint_list)

# Also extract body positions for speed computation
df_pos = extract_body_positions(ik_data['xpos'], ik_data['names_xpos'], ['thorax'])
df = df.merge(df_pos, on='frame')

print(f"\nExtracted {len(df)} frames with {len(df.columns)} columns")
df.head()

In [ ]:
# Get joint angle columns (excluding frame and position columns)
joint_cols = [f"{leg}_{joint}" for leg, joint, _ in joint_list]

# Compute derivatives
df = compute_derivatives(df, joint_cols, fps=FPS)

# Compute phases (using tibia as reference)
df = compute_phases(df, LEGS, joint='tibia', fps=FPS)

# Classify swing/stance
df = classify_swing_stance(df, LEGS)

# Compute n_legs_stance
df = compute_n_legs_stance(df, LEGS)

# Compute speed
df = compute_speed(df, body='thorax', fps=FPS)

print(f"\nPreprocessed DataFrame: {len(df.columns)} columns")
print(f"Columns: {list(df.columns)[:10]}...")

In [ ]:
# Build feature matrix
X, feature_names, valid_mask = get_feature_matrix(df, joint_list, include_derivatives=True)

# Get aligned DataFrame for plotting
df_valid = df.loc[valid_mask].reset_index(drop=True)

In [ ]:
# Run PCA
pca_result, pca, scaler = run_pca(X, n_components=10)

In [ ]:
# Add PC values to DataFrame
for i in range(pca_result.shape[1]):
    df_valid[f'PC{i+1}'] = pca_result[:, i]

---

## 8. Visualizations

In [ ]:
# PCA Variance plot (Scree plot)
fig, ax = plt.subplots(figsize=(8, 5))
plot_pca_variance(pca, ax=ax)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_variance.pdf')
plt.show()

In [ ]:
# PCA Loadings heatmap
fig, ax = plot_pca_loadings(pca, feature_names, joint_list, n_pcs=4)
plt.savefig(OUTPUT_DIR / 'pca_loadings.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# PCA trajectory colored by time
fig, ax = plt.subplots(figsize=(10, 8))
plot_pca_trajectory(pca_result, df_valid, color_by='frame', ax=ax)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_trajectory_time.pdf')
plt.show()

In [ ]:
# PCA trajectory colored by n_legs_stance
if 'n_legs_stance' in df_valid.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    plot_pca_trajectory(pca_result, df_valid, color_by='n_legs_stance', ax=ax)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'pca_trajectory_nlegs.pdf')
    plt.show()

In [ ]:
# Joint vs phase for each leg (using tibia as example)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, leg in zip(axes.flat, LEGS):
    plot_joint_vs_phase(df_valid, leg, 'tibia', ax=ax)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'tibia_vs_phase.pdf')
plt.show()

In [ ]:
# ALL JOINTS vs PHASE for ALL LEGS (7 joints × 6 legs grid)
# This shows how each joint angle varies with step phase across all legs

MAIN_JOINTS = ['coxa_flexion', 'coxa_twist', 'coxa', 'femur_twist', 'femur', 'tibia', 'tarsus']
n_joints = len(MAIN_JOINTS)
n_legs = len(LEGS)
n_bins = 24

fig, axes = plt.subplots(n_joints, n_legs, figsize=(16, 18), sharex=True)

for i, joint in enumerate(MAIN_JOINTS):
    for j, leg in enumerate(LEGS):
        ax = axes[i, j]
        
        col = f"{leg}_{joint}"
        phase_col = f"{leg}_phase"
        
        if col not in df_valid.columns or phase_col not in df_valid.columns:
            ax.set_visible(False)
            continue
        
        # Get valid data
        valid = df_valid[[col, phase_col]].dropna()
        angles = valid[col].values
        phases = valid[phase_col].values
        
        if len(angles) < 10:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue
        
        # Bin by phase
        phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
        bin_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
        bin_indices = np.digitize(phases, phase_bins) - 1
        bin_indices = np.clip(bin_indices, 0, n_bins - 1)
        
        # Compute mean and std per bin
        bin_means = np.array([angles[bin_indices == k].mean() if np.any(bin_indices == k) else np.nan 
                             for k in range(n_bins)])
        bin_stds = np.array([angles[bin_indices == k].std() if np.any(bin_indices == k) else np.nan 
                            for k in range(n_bins)])
        
        # Plot
        ax.fill_between(bin_centers, bin_means - bin_stds, bin_means + bin_stds, alpha=0.3, color='blue')
        ax.plot(bin_centers, bin_means, 'b-', lw=1.5)
        ax.axvline(0, color='r', linestyle='--', alpha=0.5, lw=0.5)
        
        ax.set_xlim(-np.pi, np.pi)
        
        # Labels
        if i == n_joints - 1:
            ax.set_xlabel(LEG_LABELS.get(leg, leg))
        if j == 0:
            ax.set_ylabel(JOINT_LABELS.get(joint, joint), fontsize=9)
        
        # Column titles (leg names) only on top row
        if i == 0:
            ax.set_title(LEG_LABELS.get(leg, leg), fontsize=10)

plt.suptitle('All Joints vs Step Phase (mean ± std)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'all_joints_vs_phase_full.pdf', bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'all_joints_vs_phase_full.pdf'}")

In [ ]:
# Inter-leg phase relationships
fig, axes = plot_leg_coordination(df_valid, LEGS)
plt.savefig(OUTPUT_DIR / 'leg_coordination.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Compute tripod coordination strength
df_valid, tcs_results = compute_tripod_coordination_strength(df_valid, LEGS)

---

## 9. Optional: UMAP

In [ ]:
if HAS_UMAP:
    umap_result, reducer, _ = run_umap(X, n_neighbors=15, min_dist=0.1)
    
    if umap_result is not None:
        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(umap_result[:, 0], umap_result[:, 1], 
                            c=df_valid['n_legs_stance'], cmap='viridis', 
                            s=5, alpha=0.5)
        ax.set_xlabel('UMAP 1')
        ax.set_ylabel('UMAP 2')
        ax.set_title('UMAP Embedding (colored by n_legs_stance)')
        plt.colorbar(scatter, ax=ax)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'umap_nlegs.pdf')
        plt.show()
else:
    print("UMAP not available. Skipping.")

---

## 10. MuJoCo Visualization

Visualize joint angles on the actual fly model with interactive frame scrubbing.

In [ ]:
# MuJoCo setup
import mujoco
from ipywidgets import interact, IntSlider
from IPython.display import display, clear_output

# MuJoCo model path
MUJOCO_MODEL_PATH = '/home/user/src/Fly_tracking/assets/fruitfly_v1/fruitfly_v1_free.xml'

# Load model and data
mj_model = mujoco.MjModel.from_xml_path(MUJOCO_MODEL_PATH)
mj_data = mujoco.MjData(mj_model)

# Get qpos from IK data
qpos = ik_data['qpos']
names_qpos = ik_data['names_qpos']

print(f"MuJoCo model loaded: {MUJOCO_MODEL_PATH}")
print(f"  qpos size: {mj_model.nq}")
print(f"  IK qpos shape: {qpos.shape}")

In [ ]:
def render_frame_with_angles(frame_idx, mj_model, mj_data, qpos, names_qpos, 
                              width=800, height=600, show_angles=True):
    """
    Render a single frame with the fly model and optionally display key joint angles.
    
    Args:
        frame_idx: Frame index to render
        mj_model: MuJoCo model
        mj_data: MuJoCo data
        qpos: Array of joint positions (N, nq)
        names_qpos: List of joint names
        width, height: Render resolution
        show_angles: Whether to display joint angle values
    
    Returns:
        Rendered image as numpy array
    """
    # Set joint positions for this frame
    mj_data.qpos[:] = qpos[frame_idx]
    mujoco.mj_forward(mj_model, mj_data)
    
    # Create renderer
    renderer = mujoco.Renderer(mj_model, height=height, width=width)
    
    # Set up camera for side view
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat[:] = mj_data.body('thorax').xpos
    cam.distance = 6.0
    cam.azimuth = 90  # Side view
    cam.elevation = -15
    
    renderer.update_scene(mj_data, camera=cam)
    img_side = renderer.render()
    
    # Top view
    cam.azimuth = 0
    cam.elevation = -90
    renderer.update_scene(mj_data, camera=cam)
    img_top = renderer.render()
    
    renderer.close()
    
    return img_side, img_top


def interactive_frame_viewer(frame_idx):
    """Interactive viewer function for ipywidgets."""
    clear_output(wait=True)
    
    # Render the frame
    img_side, img_top = render_frame_with_angles(frame_idx, mj_model, mj_data, qpos, names_qpos)
    
    # Create figure with two views + joint angles table
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Side view
    axes[0].imshow(img_side)
    axes[0].set_title(f'Side View - Frame {frame_idx}')
    axes[0].axis('off')
    
    # Top view
    axes[1].imshow(img_top)
    axes[1].set_title('Top View')
    axes[1].axis('off')
    
    # Joint angles table
    axes[2].axis('off')
    
    # Get key joint angles for one leg (T1_left as example)
    display_joints = ['coxa_flexion', 'coxa', 'femur', 'tibia']
    table_data = []
    for leg in ['T1_left', 'T1_right']:
        leg_label = LEG_LABELS.get(leg, leg)
        for joint in display_joints:
            full_name = f"{joint}_{leg}"
            if full_name in names_qpos:
                idx = names_qpos.index(full_name)
                angle = np.degrees(qpos[frame_idx, idx])
                table_data.append([f'{leg_label} {JOINT_LABELS.get(joint, joint)}', f'{angle:.1f}°'])
    
    table = axes[2].table(cellText=table_data,
                          colLabels=['Joint', 'Angle'],
                          loc='center',
                          cellLoc='left')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.5)
    axes[2].set_title('Front Leg Angles')
    
    plt.tight_layout()
    plt.show()
    
    # Print time info
    time_ms = frame_idx / FPS * 1000
    print(f"Frame {frame_idx} | Time: {time_ms:.1f} ms | {frame_idx}/{len(qpos)-1}")


print("Interactive viewer function defined.")

---

## Summary

This notebook provides tools to:
1. Load joint angles from IK H5 files
2. Compute derivatives and step phases
3. Run PCA/UMAP for dimensionality reduction
4. Visualize joint coordination patterns (tibia vs phase, all joints vs phase grid)
5. **Interactive MuJoCo visualization** - scrub through frames to see joint angles on the fly model

**Next steps**:
- Load specific walking bouts detected in Sandbox_Strict
- Compare coordination across different speeds
- Analyze differences between individual legs
- Compare across multiple flies/sessions

---

## Summary

This notebook provides tools to:
1. Load joint angles from IK H5 files
2. Compute derivatives and step phases
3. Run PCA/UMAP for dimensionality reduction
4. Visualize joint coordination patterns

**Next steps**:
- Load specific walking bouts detected in Sandbox_Strict
- Compare coordination across different speeds
- Analyze differences between individual legs
- Compare across multiple flies/sessions